# 第三十课:PagedAttention——像操作系统一样管理 KV

讲义:[30-paged-attention.md](30-paged-attention.md)。分页 KV 分配器手撕 + 分页注意力逐位等价验证 + 前缀共享账;块字节数实读 SmolLM2-135M config。

In [ ]:
# ===== 块管理器(讲义 §2) =====
import glob
import json
import struct
import numpy as np

BLOCK = 16          # vLLM 默认块大小(token)

class BlockManager:
    def __init__(self, n_blocks):
        self.free = list(range(n_blocks))     # 空闲物理块
        self.refcnt = {}                      # 物理块 -> 引用计数(共享)

    def alloc(self):
        return self.free.pop()

    def free_block(self, b):
        self.refcnt[b] = self.refcnt.get(b, 1) - 1
        if self.refcnt[b] <= 0:               # 没人引用了才真还
            self.free.append(b)
            del self.refcnt[b]

    def share(self, b):
        self.refcnt[b] = self.refcnt.get(b, 1) + 1
        return b

In [ ]:
# ===== E1:碎片率仿真(讲义 §3) =====
def sim_fragmentation(seed=0, n_req=50, max_seq=8192):
    rng = np.random.default_rng(seed)
    reqs = []
    for i in range(n_req):
        final = int(rng.integers(64, 1501))
        cancelled = rng.random() < 0.2
        used = int(final * rng.uniform(0.2, 0.8)) if cancelled else final
        reqs.append((final, used))
    cont_needed = sum(max_seq for _ in reqs)           # 连续:按最大长度预留
    used = sum(u for _, u in reqs)
    paged_needed = sum((u + BLOCK - 1) // BLOCK for _, u in reqs) * BLOCK
    return dict(cont_util=used / cont_needed, paged_util=used / paged_needed,
                cont_total=cont_needed, paged_total=paged_needed)

for seed in (0, 1):
    r = sim_fragmentation(seed)
    print(f"seed={seed}: 连续预留利用率 {r['cont_util']:.1%} | 分页 {r['paged_util']:.1%}"
          f" | 池占用 {r['paged_total']:,} / {r['cont_total']:,} token"
          f" = {r['paged_total']/r['cont_total']:.0%}")

In [ ]:
# ===== E2:分页注意力逐位等价(讲义 §4) =====
def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

def attention_contiguous(q, k_all):
    s = k_all @ q / np.sqrt(len(q))
    return softmax(s) @ k_all

def attention_paged(q, block_table, kv_blocks):
    """block_table:逻辑块序号;kv_blocks:物理块池 [n_blocks, BLOCK, d]。"""
    k_all = np.concatenate([kv_blocks[b] for b in block_table], 0)   # gather
    return attention_contiguous(q, k_all)

rng = np.random.default_rng(0)
d = 32
kv_pool = rng.normal(size=(8, BLOCK, d))
q = rng.normal(size=d)
order = [3, 0, 6, 1]                          # 逻辑 → 物理(乱序)
logical = np.concatenate([kv_pool[b] for b in order], 0)
print("分页 vs 连续 max|diff| =",
      np.abs(attention_paged(q, order, kv_pool) - attention_contiguous(q, logical)).max())

In [ ]:
# ===== E3:前缀共享 + E4:真实模型块字节(讲义 §5–6) =====
def blocks_of(n, block=BLOCK):
    return (n + block - 1) // block

sys_len, tail, n_req = 200, 300, 8
no_paging = n_req * blocks_of(sys_len + tail)
paging = blocks_of(sys_len) + n_req * blocks_of(tail)
print(f"前缀共享:无共享 {no_paging} 块 → 共享 {paging} 块(省 {1-paging/no_paging:.0%})")

SNAP = glob.glob("/Volumes/Sam-APFS/huggingface/hub/models--HuggingFaceTB--"
                 "SmolLM2-135M-Instruct/snapshots/*")[0]
cfg = json.load(open(f"{SNAP}/config.json", encoding="utf-8"))
L = cfg["num_hidden_layers"]
kv_heads = cfg.get("num_key_value_heads", cfg["num_attention_heads"])
hd = cfg["hidden_size"] // cfg["num_attention_heads"]
blk_bytes = BLOCK * 2 * L * kv_heads * hd * 2
print(f"实读 SmolLM2-135M:{L} 层,kv_heads={kv_heads},head_dim={hd}")
print(f"一块({BLOCK} tok)KV = {blk_bytes/1024:.0f} KB;一条 2K 上下文 = "
      f"{blk_bytes*(2048//BLOCK)/1024**3:.2f} GB")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): BlockManager 取消请求的释放轨迹打印

# TODO(C3): 手撕 CoW:共享块被追加时复制,验证原共享者 KV 不变

# TODO(D1): Qwen3-4B 的块字节与 32K 上下文账(36层 GQA-8 head_dim 128)

# TODO(E1): 块大小 {4,8,16,64} 扫描利用率与块表长度
